In [0]:
%pip install transformers datasets torch scikit-learn pandas accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [0]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [0]:
import torch
print(torch.cuda.is_available())  # should be True
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce GTX 1050


In [0]:
# %pip install transformers datasets "huggingface_hub<1.0" torch scikit-learn pandas 'accelerate>=0.26.0'

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'accelerate"

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [0]:
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA Available: True
GPU: NVIDIA GeForce GTX 1050


In [0]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
        num_rows: 61765
    })
})


In [0]:
train_df = pd.DataFrame(dataset['train'])
# train_df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


In [0]:
def filter_missing_labels(example):
    return example['type'] is not None

dataset = dataset.filter(filter_missing_labels)

In [0]:
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

In [0]:
df = pd.DataFrame(dataset["train"])

min_count = df["type"].value_counts().min()

balanced_df = df.groupby("type").apply(
    lambda x: x.sample(n=min_count, random_state=42)
).reset_index(drop=True)

dataset["train"] = Dataset.from_pandas(balanced_df)

print("Balanced Data:\n", balanced_df["type"].value_counts())

C:\Users\issac\AppData\Local\Temp\ipykernel_6292\177240458.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby("type").apply(


Balanced Data:
 type
Change      4007
Incident    4007
Problem     4007
Request     4007
Name: count, dtype: int64


In [0]:
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(4000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(800))

In [0]:
train_df = pd.DataFrame(dataset['train'])

train_df = train_df[train_df['type'].notna()]
train_df['type'] = train_df['type'].astype(str)

labels = sorted(train_df['type'].unique().tolist())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'Change': 0, 'Incident': 1, 'Problem': 2, 'Request': 3}


In [0]:
def preprocess(example):
    subject = example.get('subject') or ""
    body = example.get('body') or ""
    
    label = example.get('type')
    if label is None:
        label = "unknown"
    
    label = str(label)
    
    example['text'] = f"Subject: {subject} Body: {body}".strip()
    example['label'] = label2id.get(label, 0)
    
    return example

dataset = dataset.map(preprocess)

Map: 100%|██████████| 9718/9718 [00:02<00:00, 4612.68 examples/s]


In [0]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [0]:
def tokenize_function(example):
    return tokenizer(
        example['text'],
        padding="max_length",
        truncation=True,
        max_length=128   # to speed up
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16028 [00:00<?, ? examples/s]

Map: 100%|██████████| 9718/9718 [00:01<00:00, 7221.56 examples/s]


In [0]:
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [0]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [0]:
def compute_metrics(eval_pred):
    logits, labels_ = eval_pred
    predictions = logits.argmax(axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_, predictions, average='weighted'
    )
    acc = accuracy_score(labels_, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [0]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",   #skip eval during training
    learning_rate=2e-5,
    per_device_train_batch_size=8,   # faster on GPU
    num_train_epochs=3,             
    logging_dir="./logs",
    fp16=False 
)

c:\Users\issac\Documents\Next GenAI assignment\venv\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [0]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    # eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [0]:
trainer.train()

  8%|▊         | 500/6012 [04:30<1:07:09,  1.37it/s]

{'loss': 0.6714, 'grad_norm': 2.2262983322143555, 'learning_rate': 1.8343313373253494e-05, 'epoch': 0.25}


 17%|█▋        | 1000/6012 [11:27<1:10:08,  1.19it/s]

{'loss': 0.4803, 'grad_norm': 2.180278778076172, 'learning_rate': 1.66833000665336e-05, 'epoch': 0.5}


 25%|██▍       | 1500/6012 [20:01<1:30:33,  1.20s/it]

...

100%|█████████▉| 6000/6012 [1:35:27<00:15,  1.32s/it]

{'loss': 0.2763, 'grad_norm': 2.0807557106018066, 'learning_rate': 5.655355954757153e-08, 'epoch': 2.99}


100%|██████████| 6012/6012 [1:35:43<00:00,  1.05it/s]

{'train_runtime': 5743.3422, 'train_samples_per_second': 8.372, 'train_steps_per_second': 1.047, 'train_loss': 0.3790901943753738, 'epoch': 3.0}


TrainOutput(global_step=6012, training_loss=0.3790901943753738, metrics={'train_runtime': 5743.3422, 'train_samples_per_second': 8.372, 'train_steps_per_second': 1.047, 'total_flos': 1592447395295232.0, 'train_loss': 0.3790901943753738, 'epoch': 3.0})

In [0]:
trainer.evaluate(tokenized_datasets["test"])

100%|██████████| 1215/1215 [03:41<00:00,  5.48it/s]


{'eval_loss': 0.5081830620765686,
 'eval_accuracy': 0.7961514714961926,
 'eval_f1': 0.8002701569170358,
 'eval_precision': 0.8147892387029203,
 'eval_recall': 0.7961514714961926,
 'eval_runtime': 221.8912,
 'eval_samples_per_second': 43.796,
 'eval_steps_per_second': 5.476,
 'epoch': 3.0}

In [0]:
import torch.nn.functional as F

def predict(subject, body):
    text = f"Subject: {subject} Body: {body}"
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    
    predicted_class_id = probs.argmax().item()
    
    return {
        "prediction": id2label[predicted_class_id],
        "confidence": round(probs.max().item(), 3),
        "all_probs": {
            id2label[i]: round(p.item(), 3) for i, p in enumerate(probs[0])
        }
    }

In [0]:
#Incident
print(predict("Login issue", "Cannot login"))
print(predict("Late delivery", "My package is delayed"))
print(predict("Damaged product", "The item arrived broken"))

{'prediction': 'Problem', 'confidence': 0.5, 'all_probs': {'Change': 0.001, 'Incident': 0.499, 'Problem': 0.5, 'Request': 0.0}}
{'prediction': 'Incident', 'confidence': 0.852, 'all_probs': {'Change': 0.001, 'Incident': 0.852, 'Problem': 0.146, 'Request': 0.0}}
{'prediction': 'Incident', 'confidence': 0.966, 'all_probs': {'Change': 0.001, 'Incident': 0.966, 'Problem': 0.032, 'Request': 0.001}}


In [0]:
# Problem
print(predict("Slow system", "The system is consistently slow every day"))
print(predict("Recurring bug", "This issue keeps happening after every update"))
print(predict("Performance issue", "Application performance has degraded over time"))

{'prediction': 'Problem', 'confidence': 0.939, 'all_probs': {'Change': 0.001, 'Incident': 0.059, 'Problem': 0.939, 'Request': 0.001}}
{'prediction': 'Problem', 'confidence': 0.61, 'all_probs': {'Change': 0.001, 'Incident': 0.389, 'Problem': 0.61, 'Request': 0.0}}
{'prediction': 'Incident', 'confidence': 0.525, 'all_probs': {'Change': 0.001, 'Incident': 0.525, 'Problem': 0.474, 'Request': 0.0}}


In [0]:
# Request
print(predict("Access request", "Please provide access to the reporting dashboard"))
print(predict("Data export","Can you provide an option to download reports in CSV format"))
print(predict("Password reset", "I need to reset my account password"))

{'prediction': 'Request', 'confidence': 0.521, 'all_probs': {'Change': 0.424, 'Incident': 0.023, 'Problem': 0.032, 'Request': 0.521}}
{'prediction': 'Request', 'confidence': 0.997, 'all_probs': {'Change': 0.001, 'Incident': 0.001, 'Problem': 0.001, 'Request': 0.997}}
{'prediction': 'Change', 'confidence': 0.991, 'all_probs': {'Change': 0.991, 'Incident': 0.004, 'Problem': 0.003, 'Request': 0.001}}


In [0]:
# Change
print(predict("Update config", "Please change system configuration settings"))
print(predict("Upgrade system", "We need to upgrade to the latest version"))
print(predict("Modify workflow", "Request to modify approval workflow"))

{'prediction': 'Change', 'confidence': 0.999, 'all_probs': {'Change': 0.999, 'Incident': 0.0, 'Problem': 0.0, 'Request': 0.0}}
{'prediction': 'Change', 'confidence': 0.999, 'all_probs': {'Change': 0.999, 'Incident': 0.0, 'Problem': 0.0, 'Request': 0.0}}
{'prediction': 'Change', 'confidence': 0.999, 'all_probs': {'Change': 0.999, 'Incident': 0.0, 'Problem': 0.0, 'Request': 0.0}}
